# 06a · Validity of Petrosian magnitudes: input lists for visual inspection

For galaxies near the cluster redshift, a Petrosian magnitude much brighter than the fiber magnitude (r_petro − r_fiber < −1) may indicate photometry contaminated by a nearby bright source. This notebook lists such objects for visual inspection and converts the result (Flag: 0 = invalid Petrosian magnitude, use the fiber magnitude; 1 = valid) into a CSV that `07_Photometry_Flag_update.ipynb` uses to replace r_petro by r_fiber where needed.

**Input**
- `A2199_mastercat_intermediate_file0.csv` – merged catalog from `03_merge_mastercat.py`

**Files**
- `06b_petromag_validity_check_SDSSimglist_input.txt` – image-list input (objid RA Dec)
- `06c_petromag_validity_check_SDSSimglist_template.txt` – result template; copied to `06c_petromag_validity_check_SDSSimglist_result.txt` and filled in by hand
- `06d_petromag_validity_check_SDSSimglist_result.csv` – the result as CSV

In [1]:
# ============================================================================
# Setup
# ============================================================================
import numpy as np
import pandas as pd

# Show every column when a DataFrame is displayed
pd.set_option('display.max_columns', None)

In [2]:
# Merged catalog (photometry + redshifts from all sources)
df = pd.read_csv('./A2199_mastercat_intermediate_file0.csv')

# Galactic-extinction-corrected model and Petrosian magnitudes (suffix "_0")
for band in ['u', 'g', 'r', 'i', 'z']:
    df[f'p_modelmag_{band}_0'] = df[f'p_modelmag_{band}'] - df[f'p_extinction_{band}']
for band in ['u', 'g', 'r', 'i', 'z']:
    df[f'p_petromag_{band}_0'] = df[f'p_petromag_{band}'] - df[f'p_extinction_{band}']
df['grmod'] = df['p_modelmag_g_0'] - df['p_modelmag_r_0']

In [47]:
# Likely members before the caustic analysis: redshift within 0.03 +/- 0.01
pseudo_member_df = df[df['z_tot_z'].between(0.03 - 0.01, 0.03 + 0.01)]

In [48]:
# Suspicious photometry: Petrosian magnitude more than 1 mag brighter than the fiber magnitude
cond = (pseudo_member_df['p_petromag_r'] - pseudo_member_df['p_fibermag_r']) < -1
target = pseudo_member_df[cond]

In [49]:
with open('06b_petromag_validity_check_SDSSimglist_input.txt', 'w') as f:
    temp = target.sort_values(by='p_modelmag_r', ascending=True)
    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']} {row['p_ra']:.6f} {row['p_dec']:.6f}\n")

In [50]:
# Template: every object starts as valid (flag 1). Copy it to
# 06c_petromag_validity_check_SDSSimglist_result.txt and edit the flags by hand; the template
# is never read back, so re-running this notebook cannot overwrite the inspection result.
with open('06c_petromag_validity_check_SDSSimglist_template.txt', 'w') as f:
    f.write("objid,petromag_r,fibermag_r,Flag\n")
    f.write("Flag 0 = Invalid Petrosian (Use fiber mag), 1 = Valid Petrosian\n")
    temp = target.sort_values(by='p_petromag_r', ascending=True)

    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']},{row['p_ra']:.6f},{row['p_dec']:.6f},{row['p_petromag_r']:.2f},{row['p_fibermag_r']:.2f},1\n")

### Inspection procedure

The inspection is done before the caustic membership analysis, so "members" here are simply galaxies with a redshift within 0.03 ± 0.01.

1. Inspect the SDSS images (0.1″/pixel) in order of Petrosian magnitude, from bright to faint.
2. List the objects that look clearly fainter than their Petrosian magnitude implies, i.e. where the fiber magnitude should be used instead.

Even for galaxies whose redshift confirms that they are real galaxies, the Petrosian magnitude can be unreliable (nearby bright source, object fainter in the image than the fiber magnitude, ...). For these objects `p_petromag_r` is replaced by `p_fibermag_r` in the master catalog.

The list below holds the candidates identified during the inspection (commented entries were examined but kept as valid).

In [ ]:
gal_fibermag_a2199members_objid_list = [
    1237659330315223520,
    1237659330315289013,
    1237659326566171345,
    1237659330852356638,
    1237659330852094660,
    # 1237659330315485821,
    # 1237659325492495126,
    # 1237659330315485760,
    # 1237655471820833890
]

In [51]:
# Filled-in result -> CSV (the first two lines are the header and a note)
file_path = "./06c_petromag_validity_check_SDSSimglist_result.txt"

vis = pd.read_csv(
    file_path,
    sep=r"\s*,\s*",
    engine="python",
    skiprows=2,
    names=["objid", "RA", "DEC", "petromag_r", "fibermag_r", "Flag"],
)

vis["Flag"] = vis["Flag"].astype(int)
vis.to_csv("./06d_petromag_validity_check_SDSSimglist_result.csv", index=False)

Among the inspected objects, those with `member = Y` and a Petrosian magnitude more than 1 mag brighter than the fiber magnitude are flagged 0: their Petrosian photometry is boosted by a neighbouring source, so the fiber magnitude is adopted instead.